Notebook 02 - Geração de Embeddings da Base Maicon

Este notebook realiza a geração dos embeddings da base documental fornecida pelo professor orientador.

Os arquivos JSON produzidos na etapa de processamento são carregados e validados antes da geração dos embeddings. Para cada chunk, o conteúdo utilizado como entrada para o modelo de embeddings segue uma estratégia de prioridade: **`cleaned_summary → summary → text`**. O campo `cleaned_summary` é utilizado sempre que disponível; na sua ausência, utiliza-se o campo `summary` e, caso ambos estejam vazios, utiliza-se o texto original (`text`) como fallback. A origem utilizada em cada embedding é registrada no campo **`embedding_source`**, permitindo a rastreabilidade dessa decisão nas etapas posteriores.

O texto original (`text`), o identificador do chunk e os demais metadados são preservados para utilização nas etapas posteriores de recuperação da informação.

Os embeddings são gerados utilizando o modelo **sentence-transformers/paraphrase-multilingual-mpnet-base-v2**, o mesmo empregado no primeiro pipeline RAG desenvolvido nesta pesquisa. Essa escolha mantém o modelo de embeddings constante entre os pipelines, permitindo que as comparações sejam realizadas sob condições experimentais equivalentes quanto à representação vetorial, variando a base documental utilizada.

Ao final da execução, os embeddings e seus respectivos metadados são armazenados na pasta **02_Embeddings**, juntamente com um manifesto que registra as características dos embeddings e a fonte textual utilizada em sua geração. Esses artefatos servem como entrada para a construção do índice vetorial FAISS no Notebook 03.

Preparação do Ambiente

In [ ]:
# Instala a biblioteca utilizada para geração dos embeddings

!pip install -q sentence-transformers

In [ ]:
# Importa as bibliotecas necessárias

import json
import pickle

import numpy as np
import pandas as pd

from pathlib import Path

from sentence_transformers import SentenceTransformer

from google.colab import drive

In [ ]:
# Monta o Google Drive

drive.mount(
    "/content/drive"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Funções Auxiliares

In [ ]:
# Define funções auxiliares para padronizar as mensagens

def print_header(title):
    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)


def print_success(message):
    print(f"\n✅ {message}")


def print_error(message):
    print(f"\n❌ {message}")


def print_warning(message):
    print(f"\n⚠️ {message}")


def print_info(label, value):
    print(f"{label:<25} {value}")

Configuração do Projeto

In [ ]:
# Define os caminhos utilizados no Notebook 02

PROJECT_PATH = Path(
    "/content/drive/MyDrive/RAG_Novo_Embeddings"
)

BASE_DIR = (
    PROJECT_PATH
    / "01_Base_Maicon"
)

EMBEDDINGS_DIR = (
    PROJECT_PATH
    / "02_Embeddings"
)

RESULTS_DIR = (
    PROJECT_PATH
    / "04_Resultados"
)

MANIFEST_FILE = (
    RESULTS_DIR
    / "manifesto_base.csv"
)

EMBEDDINGS_MANIFEST_FILE = (
    EMBEDDINGS_DIR
    / "manifesto_embeddings.csv"
)

print_header(
    "CONFIGURAÇÃO DO PROJETO"
)

print_info(
    "Projeto:",
    PROJECT_PATH
)

print_info(
    "Base Maicon:",
    BASE_DIR
)

print_info(
    "Embeddings:",
    EMBEDDINGS_DIR
)

print_info(
    "Manifesto da base:",
    MANIFEST_FILE
)

print_info(
    "Manifesto embeddings:",
    EMBEDDINGS_MANIFEST_FILE
)

print_success(
    "Caminhos configurados com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO PROJETO
Projeto:                  /content/drive/MyDrive/RAG_Novo_Embeddings
Base Maicon:              /content/drive/MyDrive/RAG_Novo_Embeddings/01_Base_Maicon
Embeddings:               /content/drive/MyDrive/RAG_Novo_Embeddings/02_Embeddings
Manifesto da base:        /content/drive/MyDrive/RAG_Novo_Embeddings/04_Resultados/manifesto_base.csv
Manifesto embeddings:     /content/drive/MyDrive/RAG_Novo_Embeddings/02_Embeddings/manifesto_embeddings.csv

✅ Caminhos configurados com sucesso.


Verificação dos Artefatos

In [ ]:
# Verifica se todos os arquivos necessários existem

print_header(
    "VERIFICAÇÃO DOS ARTEFATOS"
)

required_paths = {

    "Base Maicon": BASE_DIR,

    "Manifesto da Base": MANIFEST_FILE

}

missing = False

for name, path in required_paths.items():

    if path.exists():

        print_info(
            f"{name}:",
            "OK"
        )

    else:

        print_error(
            f"{name} não encontrado."
        )

        print_info(
            "Esperado em:",
            path
        )

        missing = True

if missing:

    raise FileNotFoundError(
        "Existem arquivos obrigatórios ausentes."
    )

print_success(
    "Todos os artefatos foram localizados."
)

print("=" * 70)


 VERIFICAÇÃO DOS ARTEFATOS
Base Maicon:              OK
Manifesto da Base:        OK

✅ Todos os artefatos foram localizados.


Leitura do Manifesto da Base

In [ ]:
# Carrega o manifesto gerado pelo Notebook 01

print_header(
    "CARREGAMENTO DO MANIFESTO"
)

manifest_df = pd.read_csv(
    MANIFEST_FILE,
    encoding="utf-8-sig"
)

print_info(
    "Artigos encontrados:",
    len(manifest_df)
)

print_info(
    "Total de chunks:",
    int(
        manifest_df[
            "total_chunks"
        ].sum()
    )
)

print_info(
    "Colunas:",
    len(
        manifest_df.columns
    )
)

print_success(
    "Manifesto carregado com sucesso."
)

print("=" * 70)


 CARREGAMENTO DO MANIFESTO
Artigos encontrados:      648
Total de chunks:          6844
Colunas:                  12

✅ Manifesto carregado com sucesso.


Construção da Base de Chunks

In [ ]:
# Carrega todos os chunks da base documental

print_header(
    "CARREGAMENTO DOS CHUNKS"
)

all_chunks = []

for _, article in manifest_df.iterrows():

    json_path = (
        BASE_DIR
        / article["json_file"]
    )

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as file:

        chunks = json.load(
            file
        )

    for chunk in chunks:

        all_chunks.append(

            {

                "article_name":
                    article["article_name"],

                "author":
                    article["author"],

                "year":
                    article["year"],

                "title":
                    article["title"],

                "chunk_id":
                    chunk["chunk_id"],

                "summary":
                    chunk.get(
                        "summary",
                        ""
                    ),

                "cleaned_summary":
                    chunk.get(
                        "cleaned_summary",
                        ""
                    ),

                "text":
                    chunk.get(
                        "text",
                        ""
                    )

            }

        )

print_info(
    "Artigos carregados:",
    len(
        manifest_df
    )
)

print_info(
    "Chunks carregados:",
    len(
        all_chunks
    )
)

print_success(
    "Base de chunks construída com sucesso."
)

print("=" * 70)


 CARREGAMENTO DOS CHUNKS
Artigos carregados:       648
Chunks carregados:        6844

✅ Base de chunks construída com sucesso.


Validação da Base de Chunks

In [ ]:
# Valida os campos essenciais de todos os chunks

print_header(
    "VALIDAÇÃO DA BASE DE CHUNKS"
)

invalid_chunks = []

for index, chunk in enumerate(
    all_chunks
):

    problems = []

    if not chunk.get(
        "article_name"
    ):
        problems.append(
            "article_name ausente"
        )

    if chunk.get(
        "chunk_id"
    ) is None:
        problems.append(
            "chunk_id ausente"
        )

    cleaned_summary = str(
        chunk.get(
            "cleaned_summary",
            ""
        )
    ).strip()

    summary = str(
        chunk.get(
            "summary",
            ""
        )
    ).strip()

    original_text = str(
        chunk.get(
            "text",
            ""
        )
    ).strip()

    # O chunk só é inválido se não houver
    # nenhuma fonte de texto disponível
    if not cleaned_summary and not summary and not original_text:

        problems.append(
            "nenhum texto disponível para embedding"
        )

    if not original_text:

        problems.append(
            "text vazio"
        )

    if problems:

        invalid_chunks.append(
            {
                "index": index,
                "article_name": chunk.get(
                    "article_name",
                    ""
                ),
                "chunk_id": chunk.get(
                    "chunk_id"
                ),
                "problems": problems
            }
        )

print_info(
    "Chunks analisados:",
    len(all_chunks)
)

print_info(
    "Chunks válidos:",
    len(all_chunks)
    - len(invalid_chunks)
)

print_info(
    "Chunks inválidos:",
    len(invalid_chunks)
)

if invalid_chunks:

    print_error(
        f"Foram encontrados {len(invalid_chunks)} chunk(s) inválido(s)."
    )

    print()

    print(
        "Primeiros problemas encontrados:\n"
    )

    for item in invalid_chunks[:5]:

        print(
            f"• Artigo: {item['article_name']}"
        )

        print(
            f"  Chunk: {item['chunk_id']}"
        )

        print(
            f"  Problemas: {', '.join(item['problems'])}"
        )

        print()

    raise ValueError(
        "A base contém chunks inválidos. "
        "Corrija os problemas antes de gerar os embeddings."
    )

print_success(
    "Todos os chunks possuem texto disponível para embedding."
)

print("=" * 70)


 VALIDAÇÃO DA BASE DE CHUNKS
Chunks analisados:        6844
Chunks válidos:           6844
Chunks inválidos:         0

✅ Todos os chunks possuem texto disponível para embedding.


Preparação dos Registros para Embeddings

In [ ]:
# Prepara os registros que serão utilizados
# para geração dos embeddings

print_header(
    "PREPARAÇÃO DOS REGISTROS PARA EMBEDDING"
)

embedding_records = []

source_counts = {
    "cleaned_summary": 0,
    "summary": 0,
    "text": 0
}

for chunk in all_chunks:

    cleaned_summary = str(
        chunk.get(
            "cleaned_summary",
            ""
        )
    ).strip()

    summary = str(
        chunk.get(
            "summary",
            ""
        )
    ).strip()

    original_text = str(
        chunk.get(
            "text",
            ""
        )
    ).strip()

    # Define a fonte utilizada para gerar o embedding
    if cleaned_summary:

        embedding_text = cleaned_summary
        embedding_source = "cleaned_summary"

    elif summary:

        embedding_text = summary
        embedding_source = "summary"

    else:

        embedding_text = original_text
        embedding_source = "text"

    # Registra a origem utilizada
    source_counts[
        embedding_source
    ] += 1

    # Mantém os metadados necessários
    # para as etapas seguintes
    record = {
        "article_name": chunk.get(
            "article_name",
            ""
        ),
        "author": chunk.get(
            "author",
            ""
        ),
        "year": chunk.get(
            "year",
            ""
        ),
        "title": chunk.get(
            "title",
            ""
        ),
        "chunk_id": chunk.get(
            "chunk_id"
        ),
        "summary": summary,
        "cleaned_summary": cleaned_summary,
        "embedding_text": embedding_text,
        "embedding_source": embedding_source,
        "original_text": original_text
    }

    embedding_records.append(
        record
    )

print_info(
    "Registros preparados:",
    len(embedding_records)
)

print()

print_info(
    "Fonte cleaned_summary:",
    source_counts["cleaned_summary"]
)

print_info(
    "Fonte summary:",
    source_counts["summary"]
)

print_info(
    "Fonte text:",
    source_counts["text"]
)

print()

print_info(
    "Total das fontes:",
    sum(source_counts.values())
)

# Verificação de consistência
if len(embedding_records) != len(all_chunks):

    raise ValueError(
        "O número de registros preparados não corresponde "
        "ao número total de chunks."
    )

if sum(source_counts.values()) != len(all_chunks):

    raise ValueError(
        "A contagem das fontes de embedding está inconsistente."
    )

# Garante que nenhum texto utilizado para embedding esteja vazio
empty_embedding_texts = sum(
    not str(
        record["embedding_text"]
    ).strip()
    for record in embedding_records
)

if empty_embedding_texts > 0:

    raise ValueError(
        f"Foram encontrados {empty_embedding_texts} registros "
        "com embedding_text vazio."
    )

# Garante que todos os registros preservem o texto original
empty_original_texts = sum(
    not str(
        record["original_text"]
    ).strip()
    for record in embedding_records
)

if empty_original_texts > 0:

    raise ValueError(
        f"Foram encontrados {empty_original_texts} registros "
        "com original_text vazio."
    )

print_success(
    "Registros preparados e validados com sucesso."
)

print("=" * 70)


 PREPARAÇÃO DOS REGISTROS PARA EMBEDDING
Registros preparados:     6844

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

Total das fontes:         6844

✅ Registros preparados e validados com sucesso.


Carregamento do Modelo de Embeddings

In [ ]:
# Carrega o modelo de embeddings

print_header(
    "CARREGAMENTO DO MODELO"
)

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

embedding_dimension = (
    embedding_model
    .get_embedding_dimension()
)

print_info(
    "Modelo:",
    EMBEDDING_MODEL_NAME
)

print_info(
    "Dimensão:",
    embedding_dimension
)

if embedding_dimension != 384:

    raise ValueError(
        "A dimensão do modelo não corresponde "
        "ao valor esperado de 384."
    )

print_success(
    "Modelo de embeddings carregado com sucesso."
)

print("=" * 70)


 CARREGAMENTO DO MODELO


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo:                   sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dimensão:                 384

✅ Modelo de embeddings carregado com sucesso.


Geração dos Embeddings

In [ ]:
# Gera os embeddings e separa vetores e metadados

print_header(
    "GERAÇÃO DOS EMBEDDINGS"
)

texts_to_embed = [
    record["embedding_text"]
    for record in embedding_records
]

embeddings_matrix = embedding_model.encode(
    texts_to_embed,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings_matrix = np.asarray(
    embeddings_matrix,
    dtype=np.float32
)

metadata_records = []

for record in embedding_records:

    metadata_records.append(
        {
            "article_name": record["article_name"],
            "author": record["author"],
            "year": record["year"],
            "title": record["title"],
            "chunk_id": record["chunk_id"],
            "embedding_text": record["embedding_text"],
            "embedding_source": record["embedding_source"],
            "original_text": record["original_text"]
        }
    )

print_info(
    "Embeddings gerados:",
    embeddings_matrix.shape[0]
)

print_info(
    "Dimensão:",
    embeddings_matrix.shape[1]
)

print_info(
    "Tipo:",
    embeddings_matrix.dtype
)

print_info(
    "Metadados:",
    len(metadata_records)
)

print()

print_info(
    "Fonte cleaned_summary:",
    sum(
        record["embedding_source"] == "cleaned_summary"
        for record in metadata_records
    )
)

print_info(
    "Fonte summary:",
    sum(
        record["embedding_source"] == "summary"
        for record in metadata_records
    )
)

print_info(
    "Fonte text:",
    sum(
        record["embedding_source"] == "text"
        for record in metadata_records
    )
)

if embeddings_matrix.shape[0] != len(metadata_records):

    raise ValueError(
        "A quantidade de embeddings não corresponde "
        "à quantidade de metadados."
    )

if embeddings_matrix.shape[1] != 384:

    raise ValueError(
        "A dimensão dos embeddings não corresponde "
        "ao valor esperado de 384."
    )

print_success(
    "Embeddings e metadados preparados com sucesso."
)

print("=" * 70)


 GERAÇÃO DOS EMBEDDINGS


Batches:   0%|          | 0/214 [00:00<?, ?it/s]

Embeddings gerados:       6844
Dimensão:                 384
Tipo:                     float32
Metadados:                6844

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

✅ Embeddings e metadados preparados com sucesso.


Validação dos Embeddings

In [ ]:
# Valida a matriz de embeddings

print_header(
    "VALIDAÇÃO DOS EMBEDDINGS"
)

embedding_norms = np.linalg.norm(
    embeddings_matrix,
    axis=1
)

print_info(
    "Quantidade de vetores:",
    embeddings_matrix.shape[0]
)

print_info(
    "Dimensão:",
    embeddings_matrix.shape[1]
)

print_info(
    "Norma mínima:",
    round(
        float(
            embedding_norms.min()
        ),
        6
    )
)

print_info(
    "Norma máxima:",
    round(
        float(
            embedding_norms.max()
        ),
        6
    )
)

print_info(
    "Norma média:",
    round(
        float(
            embedding_norms.mean()
        ),
        6
    )
)

if not np.allclose(
    embedding_norms,
    1.0,
    atol=1e-5
):

    raise ValueError(
        "Existem embeddings que não estão normalizados."
    )

print_success(
    "Todos os embeddings foram validados."
)

print("=" * 70)


 VALIDAÇÃO DOS EMBEDDINGS
Quantidade de vetores:    6844
Dimensão:                 384
Norma mínima:             1.0
Norma máxima:             1.0
Norma média:              1.0

✅ Todos os embeddings foram validados.


Salvamento dos Embeddings

In [ ]:
# Salva os embeddings e os metadados

print_header(
    "SALVAMENTO DOS EMBEDDINGS"
)

EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EMBEDDINGS_FILE = (
    EMBEDDINGS_DIR
    / "embeddings.npy"
)

METADATA_FILE = (
    EMBEDDINGS_DIR
    / "metadata.pkl"
)

np.save(
    EMBEDDINGS_FILE,
    embeddings_matrix
)

with open(
    METADATA_FILE,
    "wb"
) as file:

    pickle.dump(
        metadata_records,
        file
    )

print_info(
    "Embeddings:",
    EMBEDDINGS_FILE.name
)

print_info(
    "Metadados:",
    METADATA_FILE.name
)

print_success(
    "Arquivos salvos com sucesso."
)

print("=" * 70)


 SALVAMENTO DOS EMBEDDINGS
Embeddings:               embeddings.npy
Metadados:                metadata.pkl

✅ Arquivos salvos com sucesso.


Verificação dos Arquivos Salvos

In [ ]:
# Verifica se os arquivos salvos podem ser carregados corretamente

print_header(
    "VERIFICAÇÃO DOS ARQUIVOS SALVOS"
)

loaded_embeddings = np.load(
    EMBEDDINGS_FILE
)

with open(
    METADATA_FILE,
    "rb"
) as file:

    loaded_metadata = pickle.load(
        file
    )

print_info(
    "Embeddings carregados:",
    loaded_embeddings.shape[0]
)

print_info(
    "Dimensão:",
    loaded_embeddings.shape[1]
)

print_info(
    "Metadados carregados:",
    len(
        loaded_metadata
    )
)

if loaded_embeddings.shape[0] != len(loaded_metadata):

    raise ValueError(
        "O número de embeddings não corresponde "
        "ao número de metadados."
    )

if loaded_embeddings.shape[1] != 384:

    raise ValueError(
        "A dimensão dos embeddings está incorreta."
    )

print_success(
    "Arquivos verificados com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO DOS ARQUIVOS SALVOS
Embeddings carregados:    6844
Dimensão:                 384
Metadados carregados:     6844

✅ Arquivos verificados com sucesso.


Manifesto dos Embeddings

In [ ]:
# Cria e salva o manifesto dos embeddings

print_header(
    "CRIAÇÃO DO MANIFESTO DOS EMBEDDINGS"
)

embedding_manifest_records = []

for record in metadata_records:

    embedding_manifest_records.append(
        {
            "article_name": record["article_name"],
            "author": record["author"],
            "year": record["year"],
            "title": record["title"],
            "chunk_id": record["chunk_id"],
            "embedding_source": record["embedding_source"],
            "embedding_dimension": embeddings_matrix.shape[1],
            "embedding_text_length": len(
                record["embedding_text"]
            )
        }
    )

embeddings_manifest_df = pd.DataFrame(
    embedding_manifest_records
)

embeddings_manifest_df.to_csv(
    EMBEDDINGS_MANIFEST_FILE,
    index=False,
    encoding="utf-8-sig"
)

print_info(
    "Registros:",
    len(
        embeddings_manifest_df
    )
)

print()

print_info(
    "Fonte cleaned_summary:",
    (
        embeddings_manifest_df["embedding_source"]
        == "cleaned_summary"
    ).sum()
)

print_info(
    "Fonte summary:",
    (
        embeddings_manifest_df["embedding_source"]
        == "summary"
    ).sum()
)

print_info(
    "Fonte text:",
    (
        embeddings_manifest_df["embedding_source"]
        == "text"
    ).sum()
)

print()

print_info(
    "Arquivo:",
    EMBEDDINGS_MANIFEST_FILE.name
)

print_success(
    "Manifesto dos embeddings criado e salvo com sucesso."
)

print("=" * 70)


 CRIAÇÃO DO MANIFESTO DOS EMBEDDINGS
Registros:                6844

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

Arquivo:                  manifesto_embeddings.csv

✅ Manifesto dos embeddings criado e salvo com sucesso.


Validação dos Arquivos Gerados

In [ ]:
# Verifica se todos os arquivos gerados existem no Google Drive

print_header(
    "VALIDAÇÃO DOS ARQUIVOS GERADOS"
)

generated_files = {
    "Embeddings": EMBEDDINGS_FILE,
    "Metadados": METADATA_FILE,
    "Manifesto embeddings": EMBEDDINGS_MANIFEST_FILE
}

missing_files = []

for name, path in generated_files.items():

    if path.exists():

        print_info(
            f"{name}:",
            "OK"
        )

    else:

        print_info(
            f"{name}:",
            "AUSENTE"
        )

        missing_files.append(
            str(path)
        )

if missing_files:

    raise FileNotFoundError(
        "Existem arquivos gerados que não foram encontrados."
    )

print_success(
    "Todos os arquivos gerados foram localizados."
)

print("=" * 70)


 VALIDAÇÃO DOS ARQUIVOS GERADOS
Embeddings:               OK
Metadados:                OK
Manifesto embeddings:     OK

✅ Todos os arquivos gerados foram localizados.


Verificação Final do Notebook

In [ ]:
# Realiza a verificação final do Notebook 02

print_header(
    "VERIFICAÇÃO FINAL DO NOTEBOOK 02"
)

print_info(
    "Base Maicon:",
    "OK"
)

print_info(
    "Manifesto da base:",
    "OK"
)

print_info(
    "Modelo:",
    EMBEDDING_MODEL_NAME
)

print_info(
    "Artigos:",
    len(
        manifest_df
    )
)

print_info(
    "Chunks:",
    len(
        metadata_records
    )
)

print_info(
    "Embeddings:",
    embeddings_matrix.shape[0]
)

print_info(
    "Dimensão:",
    embeddings_matrix.shape[1]
)

print()

print_info(
    "Fonte cleaned_summary:",
    sum(
        record["embedding_source"] == "cleaned_summary"
        for record in metadata_records
    )
)

print_info(
    "Fonte summary:",
    sum(
        record["embedding_source"] == "summary"
        for record in metadata_records
    )
)

print_info(
    "Fonte text:",
    sum(
        record["embedding_source"] == "text"
        for record in metadata_records
    )
)

print()

print_info(
    "Arquivo .npy:",
    EMBEDDINGS_FILE.name
)

print_info(
    "Arquivo .pkl:",
    METADATA_FILE.name
)

print_info(
    "Manifesto embeddings:",
    EMBEDDINGS_MANIFEST_FILE.name
)

# Verificações de consistência

if len(metadata_records) != embeddings_matrix.shape[0]:

    raise ValueError(
        "A quantidade de metadados não corresponde "
        "à quantidade de embeddings."
    )

if embeddings_matrix.shape[1] != 384:

    raise ValueError(
        "A dimensão dos embeddings não corresponde "
        "ao valor esperado de 384."
    )

if sum(
    record["embedding_source"] in {
        "cleaned_summary",
        "summary",
        "text"
    }
    for record in metadata_records
) != len(metadata_records):

    raise ValueError(
        "Existem registros com embedding_source inválido."
    )

print_success(
    "Notebook 02 configurado e validado com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO FINAL DO NOTEBOOK 02
Base Maicon:              OK
Manifesto da base:        OK
Modelo:                   sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Artigos:                  648
Chunks:                   6844
Embeddings:               6844
Dimensão:                 384

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

Arquivo .npy:             embeddings.npy
Arquivo .pkl:             metadata.pkl
Manifesto embeddings:     manifesto_embeddings.csv

✅ Notebook 02 configurado e validado com sucesso.
